# Entrenamiento con MLflow Tracking \[EJERCICIO\]

Partimos del pipeline que ya construiste en `02_training_ejercicio.ipynb`:
los pasos de carga, limpieza, feature engineering, split y entrenamiento
ya están resueltos — **no los tienes que volver a implementar**.

Tu tarea en este notebook es exclusivamente aprender a usar **MLflow**:
configurar el experimento en Databricks y registrar parámetros, métricas,
el modelo y un artefacto gráfico.

Completa solo las celdas marcadas con `# TODO`.

## 1. Importar librerías

El pipeline base ya lo conoces. Solo falta importar MLflow.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# TODO: importar mlflow
# TODO: importar mlflow.sklearn  (necesario para log_model)


## 2. Configurar MLflow → Databricks \[TODO\]

`load_dotenv` lee el `.env` de la raíz y expone `DATABRICKS_HOST` y `DATABRICKS_TOKEN`
en `os.environ`. Con eso MLflow sabe a qué workspace conectarse.

In [ ]:
from dotenv import load_dotenv

load_dotenv("../.env")

DATABRICKS_HOST  = os.environ["DATABRICKS_HOST"]
DATABRICKS_TOKEN = os.environ["DATABRICKS_TOKEN"]

# TODO: decirle a MLflow que use Databricks como backend de tracking
___

# TODO: definir el nombre del experimento y activarlo
# Formato: "/Users/TU_EMAIL/nombre-experimento"
EXPERIMENT_NAME = "___"
___

print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experimento  : {EXPERIMENT_NAME}")

## 3–6. Pipeline de datos y entrenamiento

> Esta sección ya está resuelta — es la misma que construiste en `02_training_ejercicio.ipynb`.
> Ejecútala para tener el modelo listo antes de la sección de MLflow.

In [ ]:
# Cargar, limpiar y hacer feature engineering
df = pd.read_csv("../data/raw/concrete_data.csv")
df = df.drop_duplicates().dropna()

df["water_cement_ratio"] = df["water"] / (df["cement"] + 1e-9)
df["binder_total"]       = df["cement"] + df["blast_furnace_slag"] + df["fly_ash"]
df["aggregate_total"]    = df["coarse_aggregate"] + df["fine_aggregate"]

# Split
TARGET       = "concrete_compressive_strength"
TEST_SIZE    = 0.2
RANDOM_STATE = 42

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# Entrenar
N_ESTIMATORS = 100
MAX_DEPTH    = None

model = RandomForestRegressor(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)

# Métricas
y_pred = model.predict(X_test)
mae    = mean_absolute_error(y_test, y_pred)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
r2     = r2_score(y_test, y_pred)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.3f}")

## 4. Registrar el run en MLflow \[TODO\]

Todo lo que ocurra dentro del bloque `with mlflow.start_run()` queda guardado
en Databricks. Tu tarea es completar las 4 llamadas de tracking:

| Paso | Método MLflow |
|---|---|
| Registrar hiperparámetros | `mlflow.log_params(dict)` |
| Registrar métricas | `mlflow.log_metrics(dict)` |
| Registrar el modelo | `mlflow.sklearn.log_model(...)` |
| Registrar la figura | `mlflow.log_figure(fig, artifact_file=...)` |

In [ ]:
with mlflow.start_run(run_name="random_forest_baseline"):

    # -- 1. Registrar hiperparámetros -----------------------------------------
    # TODO: loguear n_estimators, max_depth, test_size y random_state
    ___

    # -- 2. Registrar métricas ------------------------------------------------
    # TODO: loguear mae, rmse y r2 (ya calculados arriba)
    ___

    # -- 3. Registrar el modelo -----------------------------------------------
    X_train_f64 = X_train.astype("float64")
    signature   = mlflow.models.infer_signature(X_train_f64, model.predict(X_train))

    # TODO: loguear el modelo con su firma y un input_example de 5 filas
    ___

    # -- 4. Registrar figura predicho vs real ---------------------------------
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(y_test, y_pred, alpha=0.6, edgecolors="k", linewidths=0.4)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, "r--", linewidth=1.5, label="Predicción perfecta")
    ax.set_xlabel("Valor real (MPa)")
    ax.set_ylabel("Predicción (MPa)")
    ax.set_title(f"Predicho vs Real  |  R²={r2:.3f}")
    ax.legend()
    plt.tight_layout()

    # TODO: loguear la figura como artefacto PNG
    ___

    plt.show()

    run_id = mlflow.active_run().info.run_id
    print(f"Run registrado  : {run_id}")
    print(f"MAE  : {mae:.3f} | RMSE : {rmse:.3f} | R²  : {r2:.3f}")